# 10 · Equity Spot, Forwards, and Options

Equity pricers leverage discount curves, spot observations, dividend yields, and volatility surfaces. This notebook covers spot valuation and plain-vanilla calls/puts.

### Learning Objectives
- Build an equity market context with spot, dividend yield, and vol surface
- Price equity spot exposure through the valuations API
- Value European calls and puts with delta/gamma/vega metrics

In [ ]:
from datetime import date

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.scalars import MarketScalar
from finstack.core.market_data.surfaces import VolSurface
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import Equity, EquityOption
from finstack.valuations.pricer import create_standard_registry

as_of = date(2024, 1, 2)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9970), (1.0, 0.9940), (3.0, 0.9725), (5.0, 0.9480)],
    )
)
market.insert_price("ACME-SPOT", MarketScalar.price(Money(150.0, USD)))
market.insert_price("ACME-DIVYIELD", MarketScalar.unitless(0.015))
market.insert_surface(
    VolSurface(
        "ACME-VOL",
        expiries=[0.25, 0.5, 1.0, 2.0],
        strikes=[120.0, 140.0, 160.0, 180.0],
        grid=[
            [0.28, 0.26, 0.25, 0.24],
            [0.27, 0.25, 0.24, 0.23],
            [0.26, 0.24, 0.23, 0.22],
            [0.25, 0.23, 0.22, 0.21],
        ],
    )
)
registry = create_standard_registry()


## 1. Spot Exposure
Use `Equity.builder(...).<methods>...build()` to define shares for a ticker. Pricing returns PV in base currency using the stored spot scalar.

In [ ]:
equity = (
    Equity.builder("ACME-SPOT")
    .ticker("ACME")
    .currency(USD)
    .shares(1_000.0)
    .price_id("ACME-SPOT")
    .div_yield_id("ACME-DIVYIELD")
    .build()
)
spot_res = registry.price(equity, "discounting", market)
print("Equity PV:", spot_res.value)


## 2. European Call & Put
Options require strike, expiry, notional, and references to spot/dividend/vol IDs. Request delta/gamma/vega for risk reporting.

In [ ]:
call = (
    EquityOption.builder("ACME-CALL-150")
    .money(Money(150.0, USD))
    .ticker("ACME")
    .strike(150.0)
    .expiry(date(2024, 12, 31))
    .contract_size(100.0)
    .option_type("call")
    .exercise_style("european")
    .disc_id("USD-OIS")
    .spot_id("ACME-SPOT")
    .vol_surface("ACME-VOL")
    .div_yield_id("ACME-DIVYIELD")
    .build()
)
call_res = registry.price_with_metrics(call, "discounting", market, ["delta", "gamma", "vega"])
print("Call PV:", call_res.value)
print("Delta:", call_res.measures.get("delta"))
print("Gamma:", call_res.measures.get("gamma"))
print("Vega:", call_res.measures.get("vega"))

put = (
    EquityOption.builder("ACME-PUT-140")
    .money(Money(140.0, USD))
    .ticker("ACME")
    .strike(140.0)
    .expiry(date(2024, 9, 30))
    .contract_size(100.0)
    .option_type("put")
    .exercise_style("european")
    .disc_id("USD-OIS")
    .spot_id("ACME-SPOT")
    .vol_surface("ACME-VOL")
    .div_yield_id("ACME-DIVYIELD")
    .build()
)
print(f"\nCreated put option: {put}")
print(f"  Ticker: {put.ticker}")
print(f"  Strike: {put.strike}")
print(f"  Expiry: {put.expiry}")


## Summary
- Equity pricers consume the same `MarketContext` aggregator as rates/credit.
- Spot exposure is simply PV = shares × observed price.
- Vanilla options consume spot, dividend yield, and vol surface IDs while emitting standard Greeks for hedging.